AJUDA CHAT PARA RESOLVER PROBLEMA

In [1]:
from si.io.csv_file import read_csv
print("Import OK!")

Import OK!


AJUDA CHAT PARA RESOLVER PROBLEMA

In [2]:
from pathlib import Path
import numpy as np
from si.io.csv_file import read_csv

iris_path = Path(r"C:\Users\filip\OneDrive\Attachments\Ambiente de Trabalho\SIB_gh\si2\datasets\iris\iris.csv")
dataset = read_csv(iris_path, sep=",", features=True, label=True)

X = dataset.X
y = dataset.y


In [3]:
from pathlib import Path
import numpy as np

from si.io.csv_file import read_csv  # ajusta se o caminho for diferente

# 1.1) Carregar iris.csv
# Ajusta o caminho se o ficheiro estiver noutro lado
iris_path = Path(r"C:\Users\filip\OneDrive\Attachments\Ambiente de Trabalho\SIB_gh\si2\datasets\iris\iris.csv")
dataset = read_csv(iris_path, sep=",", features=True, label=True)

X = dataset.X     # matriz de features
y = dataset.y     # vector de labels

# 1.2) Selecionar a penúltima variável independente e mostrar a sua dimensão
penultima = X[:, -2]   # todas as linhas, penúltima coluna
print("1.2) Dimensão da penúltima variável:", penultima.shape)

# 1.3) Últimas 10 amostras + média por feature nessas 10
last10 = X[-10:, :]    # últimas 10 linhas, todas as colunas
mean_last10 = np.mean(last10, axis=0)
print("1.3) Médias das últimas 10 amostras por feature:", mean_last10)

# 1.4) Amostras com valores <= 6 em TODAS as features
mask_le6 = np.all(X <= 6, axis=1)  # True só onde TODAS as colunas são <= 6
num_le6 = np.sum(mask_le6)
print("1.4) Nº de amostras com todas as features <= 6:", num_le6)

# 1.5) Amostras com label diferente de 'Iris-setosa'
mask_not_setosa = (y != "Iris-setosa")
num_not_setosa = np.sum(mask_not_setosa)
print("1.5) Nº de amostras com label != 'Iris-setosa':", num_not_setosa)


1.2) Dimensão da penúltima variável: (150,)
1.3) Médias das últimas 10 amostras por feature: [6.45 3.03 5.33 2.17]
1.4) Nº de amostras com todas as features <= 6: 89
1.5) Nº de amostras com label != 'Iris-setosa': 100


Exercício 2.1

In [4]:
import numpy as np

class Dataset:
    # ... resto da implementação que já lá está

    def dropna(self):
        """
        Remove todas as amostras (linhas) que tenham pelo menos um NaN em X.
        Atualiza também y, se existir. Retorna self.
        """
        mask = ~np.any(np.isnan(self.X), axis=1)  # True nas linhas sem NaN
        self.X = self.X[mask]
        if self.y is not None:
            self.y = self.y[mask]
        return self

    def fillna(self, value="mean"):
        """
        Substitui NaNs em X por:
        - valor numérico (float/int), OU
        - média ("mean") de cada feature, OU
        - mediana ("median") de cada feature.
        Retorna self.
        """
        X = self.X.astype(float).copy()

        if isinstance(value, (float, int)):
            fill_vals = np.full(X.shape[1], float(value))
        elif value == "mean":
            fill_vals = np.nanmean(X, axis=0)
        elif value == "median":
            fill_vals = np.nanmedian(X, axis=0)
        else:
            raise ValueError("value must be a float, 'mean' or 'median'")

        inds = np.where(np.isnan(X))
        X[inds] = np.take(fill_vals, inds[1])

        self.X = X
        return self

    def remove_by_index(self, index: int):
        """
        Remove a amostra com o índice dado (linha index).
        Atualiza também y, se existir. Retorna self.
        """
        n = self.X.shape[0]
        if not (-n <= index < n):
            raise IndexError("index out of range")

        # suportar índices negativos
        if index < 0:
            index = n + index

        mask = np.ones(n, dtype=bool)
        mask[index] = False

        self.X = self.X[mask]
        if self.y is not None:
            self.y = self.y[mask]

        return self


Exercício 2.2

In [5]:
# Exemplos simples de uso dos novos métodos (Exercício 2)

print("\n[Ex2] Exemplos com dropna/fillna/remove_by_index")

# Criar um dataset artificial com NaNs
X_demo = np.array([[1.0, 2.0],
                   [np.nan, 3.0],
                   [4.0, np.nan]])
y_demo = np.array([0, 1, 2])

from si.data.dataset import Dataset
demo_ds = Dataset(X=X_demo, y=y_demo, features=["f1", "f2"], label="class")

print("Original X:\n", demo_ds.X)

# fillna com média
demo_ds_fill = Dataset(X=X_demo.copy(), y=y_demo.copy(),
                       features=["f1", "f2"], label="class").fillna("mean")
print("Depois de fillna('mean'):\n", demo_ds_fill.X)

# dropna
demo_ds_drop = Dataset(X=X_demo.copy(), y=y_demo.copy(),
                       features=["f1", "f2"], label="class").dropna()
print("Depois de dropna, X:\n", demo_ds_drop.X, "\ny:", demo_ds_drop.y)

# remove_by_index
demo_ds_removed = demo_ds_drop.remove_by_index(0)
print("Depois de remove_by_index(0), X:\n", demo_ds_removed.X, "\ny:", demo_ds_removed.y)



[Ex2] Exemplos com dropna/fillna/remove_by_index
Original X:
 [[ 1.  2.]
 [nan  3.]
 [ 4. nan]]
Depois de fillna('mean'):
 [[1.  2. ]
 [2.5 3. ]
 [4.  2.5]]
Depois de dropna, X:
 [[1. 2.]] 
y: [0]
Depois de remove_by_index(0), X:
 [] 
y: []


AJUDA CHAT PARA RESOLVER PROBLEMA

In [6]:
from si.data.dataset import Dataset
import numpy as np

ds = Dataset(X=np.array([[1, np.nan],[2,3]]))
ds.fillna("mean").X

array([[1., 3.],
       [2., 3.]])

Exercício 2.3

In [7]:
import numpy as np
from si.data.dataset import Dataset

def test_dropna_removes_rows_with_nan_and_updates_y():
    X = np.array([[1.0, np.nan],
                  [2.0, 3.0],
                  [np.nan, 4.0]])
    y = np.array([0, 1, 2])
    ds = Dataset(X=X, y=y, features=["a", "b"], label="class")

    ds.dropna()

    assert ds.X.shape == (1, 2)
    assert ds.y.shape == (1,)
    # A única linha sem NaN era a do índice 1
    assert ds.y[0] == 1

def test_fillna_replaces_all_nans():
    X = np.array([[1.0, np.nan],
                  [2.0, 3.0],
                  [np.nan, 4.0]])
    ds_mean = Dataset(X=X.copy()).fillna("mean")
    assert not np.isnan(ds_mean.X).any()

    ds_median = Dataset(X=X.copy()).fillna("median")
    assert not np.isnan(ds_median.X).any()

    ds_value = Dataset(X=X.copy()).fillna(0.5)
    assert not np.isnan(ds_value.X).any()

def test_remove_by_index_removes_correct_row_and_updates_y():
    X = np.arange(12).reshape(6, 2)
    y = np.array([0, 1, 2, 3, 4, 5])
    ds = Dataset(X=X, y=y)

    ds.remove_by_index(2)

    assert ds.X.shape == (5, 2)
    assert (ds.y == np.array([0, 1, 3, 4, 5])).all()


Exercício 3

In [10]:
from pathlib import Path
from si.io.csv_file import read_csv
from si.feature_selection.select_percentile import SelectPercentile

# assumir que este script é corrido a partir da raiz do projeto
iris_path = Path(r"C:\Users\filip\OneDrive\Attachments\Ambiente de Trabalho\SIB_gh\si2\datasets\iris\iris.csv")
dataset = read_csv(iris_path, sep=",", features=True, label=True)

print("Features originais:", dataset.features)
print("Shape original:", dataset.X.shape)

selector = SelectPercentile(percentile=50)  # ex: 50% das features
selector.fit(dataset)
dataset_new = selector.transform(dataset)

print("Features selecionadas:", dataset_new.features)
print("Shape depois de SelectPercentile:", dataset_new.X.shape)



Features originais: Index(['sepal_length', 'sepal_width', 'petal_length', 'petal_width'], dtype='object')
Shape original: (150, 4)
Features selecionadas: ['petal_length', 'petal_width']
Shape depois de SelectPercentile: (150, 2)
